In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, row_number, count
from pyspark.sql.window import Window
import pandas as pd

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Tugas5_Dany_Akhdan_Imaduddin") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession berhasil dibuat!")

# 1. Membaca transaksi_tugas5.csv dari HDFS dan menambahkan kolom pendapatan
# Pastikan HDFS aktif dan path sesuai dengan lokasi penyimpanan file Anda
df_transaksi_tugas = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas5/transaksi_tugas5.csv", header=True, inferSchema=True)
df_transaksi_tugas = df_transaksi_tugas.withColumn("pendapatan", col("unit_terjual") * col("harga_satuan"))

# 2. Membuat df_target
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))

# Daftarkan Temporary View untuk pengerjaan Bagian C (Spark SQL)
df_transaksi_tugas.createOrReplaceTempView("transaksi_tugas")
df_target.createOrReplaceTempView("target_cabang")

SparkSession berhasil dibuat!


In [5]:
# A. Agregasi pendapatan per kota
df_pendapatan_kota = df_transaksi_tugas.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# Join dengan df_target dan kalkulasi pencapaian
df_perbandingan = df_pendapatan_kota.join(df_target, on="kota", how="inner")
df_perbandingan = df_perbandingan.withColumn(
    "pencapaian_persen",
    (col("total_pendapatan") / col("target_bulanan")) * 100
).orderBy(col("pencapaian_persen").desc())

print("--- A. Perbandingan Pencapaian Target ---")
df_perbandingan.show()

--- A. Perbandingan Pencapaian Target ---


[Stage 5:>                                                          (0 + 8) / 8]

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



In [6]:
# B. Agregasi dulu total pendapatan berdasarkan kota DAN kategori
df_pendapatan_kategori = df_transaksi_tugas.groupBy("kota", "kategori").agg(
    spark_sum("pendapatan").alias("total_pendapatan_kategori")
)

# Buat partisi per kota dan urutkan berdasarkan total pendapatan per kategori (descending)
window_kategori = Window.partitionBy("kota").orderBy(col("total_pendapatan_kategori").desc())

# Tambahkan ranking lalu ambil rank 1 (terlaris)
df_kategori_terlaris = df_pendapatan_kategori.withColumn(
    "rank", row_number().over(window_kategori)
).filter(col("rank") == 1).drop("rank")

print("--- B. Kategori Terlaris per Kota ---")
df_kategori_terlaris.show()

--- B. Kategori Terlaris per Kota ---
+----------+--------------------+-------------------------+
|      kota|            kategori|total_pendapatan_kategori|
+----------+--------------------+-------------------------+
|  Magelang|Kesehatan & Kecan...|                  7275000|
| Purworejo|Kesehatan & Kecan...|                 10075000|
|  Semarang|        Rumah Tangga|                 11125000|
|      Solo|Kesehatan & Kecan...|                  8425000|
|Yogyakarta|             Fashion|                 13325000|
+----------+--------------------+-------------------------+



In [7]:
# C. Tulis kueri SQL menggunakan view yang sudah diregistrasi
hasil_sql_tugas = spark.sql('''
    SELECT tr.kota, ta.pic_cabang, COUNT(tr.order_id) AS jumlah_transaksi
    FROM transaksi_tugas tr
    JOIN target_cabang ta ON tr.kota = ta.kota
    GROUP BY tr.kota, ta.pic_cabang
    ORDER BY jumlah_transaksi DESC
''')

print("--- C. Jumlah Transaksi Per Cabang Kota ---")
hasil_sql_tugas.show()

--- C. Jumlah Transaksi Per Cabang Kota ---
+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



In [11]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.


**D.**

**Berdasarkan hasil analisis, Purworejo merupakan cabang dengan kinerja terbaik. Cabang ini meraih persentase pencapaian target tertinggi dengan jumlah transaksi terbanyak, didorong oleh tingginya penjualan pada kategori Kesehatan & Kecantikan. Sebaliknya, cabang Semarang menempati peringkat terendah dan membutuhkan perhatian khusus.**